# SQL Task – Employee Salary Analytics

## Objective

Create and manage employee records across different departments using SQL.

### Requirements

- Create `departments` and `employees` tables.
- Establish a proper relationship using foreign keys.
- Insert sample employee and department data.
- Find the highest-paid employee in each department.
- Calculate the average salary for each department.
- Find employees earning more than their department's average salary.
- Use proper data types and SQL best practices.

In [2]:
import sqlite3
import pandas as pd

In [3]:
# Create database connection
connection = sqlite3.connect("workforce.db")

# Create cursor
cursor = connection.cursor()

print("Database connected successfully!")

Database connected successfully!


In [4]:
# Enable foreign key support
cursor.execute("PRAGMA foreign_keys = ON")

# Create departments table
cursor.execute("""
CREATE TABLE IF NOT EXISTS departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL UNIQUE,
    location TEXT
)
""")

# Create employees table
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    emp_id INTEGER PRIMARY KEY,
    emp_name TEXT NOT NULL,
    dept_id INTEGER,
    salary INTEGER NOT NULL,
    hire_date TEXT,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
)
""")

connection.commit()

print("Tables created successfully!")

Tables created successfully!


In [5]:
departments_data = [
    (1, "Engineering", "Bangalore"),
    (2, "Sales", "Mumbai"),
    (3, "HR", "Delhi")
]

cursor.executemany("""
INSERT OR IGNORE INTO departments
(dept_id, dept_name, location)
VALUES (?, ?, ?)
""", departments_data)

connection.commit()

print("Departments inserted successfully!")

Departments inserted successfully!


In [6]:
employees_data = [
    (1, "Amit Sharma", 1, 85000, "2022-03-15"),
    (2, "Priya Patel", 1, 92000, "2021-07-20"),
    (3, "Rahul Verma", 2, 65000, "2023-01-10"),
    (4, "Sneha Gupta", 2, 78000, "2022-09-05"),
    (5, "Vikram Singh", 3, 55000, "2023-05-12"),
    (6, "Ananya Rao", 1, 88000, "2020-11-30")
]

cursor.executemany("""
INSERT OR IGNORE INTO employees
(emp_id, emp_name, dept_id, salary, hire_date)
VALUES (?, ?, ?, ?, ?)
""", employees_data)

connection.commit()

print("Employees inserted successfully!")

Employees inserted successfully!


In [7]:
pd.read_sql_query("""
SELECT * FROM departments
""", connection)

,dept_id,dept_name,location
0,1,Engineering,Bangalore
1,2,Sales,Mumbai
2,3,HR,Delhi


In [8]:
pd.read_sql_query("""
SELECT * FROM employees
""", connection)

,emp_id,emp_name,dept_id,salary,hire_date
0,1,Amit Sharma,1,85000,2022-03-15
1,2,Priya Patel,1,92000,2021-07-20
2,3,Rahul Verma,2,65000,2023-01-10
3,4,Sneha Gupta,2,78000,2022-09-05
4,5,Vikram Singh,3,55000,2023-05-12
5,6,Ananya Rao,1,88000,2020-11-30


In [9]:
query1 = """
SELECT
    d.dept_name,
    e.emp_name,
    e.salary
FROM employees e
JOIN departments d
    ON e.dept_id = d.dept_id
WHERE (e.dept_id, e.salary) IN (
    SELECT
        dept_id,
        MAX(salary)
    FROM employees
    GROUP BY dept_id
)
"""

pd.read_sql_query(query1, connection)

,dept_name,emp_name,salary
0,Engineering,Priya Patel,92000
1,Sales,Sneha Gupta,78000
2,HR,Vikram Singh,55000


In [10]:
query2 = """
SELECT
    d.dept_name,
    ROUND(AVG(e.salary), 2) AS avg_salary
FROM employees e
JOIN departments d
    ON e.dept_id = d.dept_id
GROUP BY d.dept_name
ORDER BY avg_salary DESC
"""

pd.read_sql_query(query2, connection)

,dept_name,avg_salary
0,Engineering,88333.33
1,Sales,71500.00
2,HR,55000.00


In [11]:
query3 = """
SELECT
    e.emp_name,
    d.dept_name,
    e.salary,
    ROUND(avg_sal.avg_salary, 2) AS dept_avg
FROM employees e
JOIN departments d
    ON e.dept_id = d.dept_id
JOIN (
    SELECT
        dept_id,
        AVG(salary) AS avg_salary
    FROM employees
    GROUP BY dept_id
) avg_sal
    ON e.dept_id = avg_sal.dept_id
WHERE e.salary > avg_sal.avg_salary
"""

pd.read_sql_query(query3, connection)

,emp_name,dept_name,salary,dept_avg
0,Priya Patel,Engineering,92000,88333.33
1,Sneha Gupta,Sales,78000,71500.00


In [12]:
connection.close()

print("Database connection closed successfully!")

Database connection closed successfully!
